# Построение финансовой модели v2

**Использование:** задайте параметры → Run All

Порядок: препроцессор → модель → сохранение в БД

In [ ]:
import sys
from pathlib import Path

# Определяем корень проекта и добавляем движок в path
NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parents[2]   
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'ROOT:   {ROOT}')
print(f'ENGINE: {ENGINE_PATH} (exists={ENGINE_PATH.exists()})')

In [ ]:
# ======================================================================
# ПАРАМЕТРЫ — настройте перед запуском
# ======================================================================

COMPANY_ID    = 'us_steel'
SCENARIO_NAME = 'base'

DB_PATH     = ROOT / 'data_mart_v2.db'
CONFIG_PATH = ROOT / 'companies' / COMPANY_ID / 'configs' / 'project.yaml'

RUN_PREPROCESSOR = True   # пересчитать EWA-драйверы из истории
RUN_MODEL        = True   # запустить 3-Statement прогноз
RUN_STRESS       = True   # стресс-сценарии (steel_downturn, rate_spike, wc_stress)
RUN_RATING       = True   # кредитный рейтинг (S&P методология)
RUN_COVENANTS    = True   # ковенанты (ICR, ND/EBITDA; авто-триггер breach stress)

print(f'Компания:   {COMPANY_ID}')
print(f'Сценарий:   {SCENARIO_NAME}')
print(f'БД:         {DB_PATH.name} (exists={DB_PATH.exists()})')
print(f'Config:     {CONFIG_PATH.name} (exists={CONFIG_PATH.exists()})')
print(f'Модули:     Preprocessor={RUN_PREPROCESSOR} Model={RUN_MODEL} Stress={RUN_STRESS} Rating={RUN_RATING} Covenants={RUN_COVENANTS}')


In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s - %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

from engine.orchestrator import build_model

result = build_model(
    company_id=COMPANY_ID,
    scenario_name=SCENARIO_NAME,
    db_path=DB_PATH,
    config_path=CONFIG_PATH,
    run_preprocessor=RUN_PREPROCESSOR,
    run_model=RUN_MODEL,
    run_stress=RUN_STRESS,
    run_rating=RUN_RATING,
    run_covenants=RUN_COVENANTS,
)

print()
if result.success:
    mr = result.model_result
    print('OK | BS diff:', max(mr.bs_diffs.values(), default=0),
          '| CF diff:', max(mr.cf_diffs.values(), default=0))
    print(f'Стресс: {list(result.stress_results.keys())}')
    if result.rating_result:
        r25 = result.rating_result.ratings.get(2025, {}).get('rating', 'N/A')
        print(f'Рейтинг 2025: {r25}')
    if result.covenants_result:
        nb_br = len(result.covenants_result.breaches())
        print(f'Ковенанты: {nb_br} нарушений')
        if 'covenant_breach' in result.stress_results:
            print('  [covenant_breach stress auto-triggered]')
else:
    print('FAIL:')
    for e in result.errors:
        print(f'  {e}')


## Income Statement ($M)

In [ ]:
import pandas as pd

if not result.success or not result.model_result:
    print('Модель не запущена')
else:
    years = sorted(result.model_result.years.keys())
    IS_ITEMS = [
        ('Revenue',              lambda s: s.revenue),
        ('COGS',                 lambda s: s.cogs),
        ('Gross Profit',         lambda s: s.gross_profit),
        ('  gross margin %',     lambda s: s.gross_profit/s.revenue*100),
        ('SGA',                  lambda s: s.sga),
        ('Earnings investees',   lambda s: s.earnings_from_investees),
        ('EBITDA',               lambda s: s.ebitda),
        ('  EBITDA margin %',    lambda s: s.ebitda/s.revenue*100),
        ('D&A',                  lambda s: -s.total_da),
        ('EBIT',                 lambda s: s.ebit),
        ('Interest expense',     lambda s: -s.interest_expense),
        ('Interest income',      lambda s: s.interest_income),
        ('Net periodic benefit', lambda s: s.net_periodic_benefit),
        ('Other fin costs',      lambda s: -s.other_financial_costs),
        ('EBT',                  lambda s: s.ebt),
        ('Tax expense',          lambda s: s.tax_expense),
        ('Net Income',           lambda s: s.net_income),
        ('  NI margin %',        lambda s: s.net_income/s.revenue*100),
    ]
    rows = []
    for name, fn in IS_ITEMS:
        row = {'': name}
        for yr in years:
            v = fn(result.model_result.years[yr])
            row[yr] = f'{v:.1f}%' if '%' in name else f'{v/1e6:.0f}'
        rows.append(row)
    display(pd.DataFrame(rows).set_index(''))

## Balance Sheet ($M)

In [ ]:
if result.success and result.model_result:
    BS_ITEMS = [
        ('── ASSETS ──────────────', lambda s: None),
        ('Cash',                    lambda s: s.cash),
        ('Accounts Receivable',     lambda s: s.accounts_receivable),
        ('Inventory',               lambda s: s.inventory),
        ('Other CA',                lambda s: s.other_ca),
        ('Total Current Assets',    lambda s: s.total_ca),
        ('PPE net',                 lambda s: s.ppe_net),
        ('ROU Asset',               lambda s: s.rou_asset),
        ('Intangibles',             lambda s: s.intangibles),
        ('Goodwill',                lambda s: s.goodwill),
        ('DTA',                     lambda s: s.dta),
        ('Other NCA',               lambda s: s.other_nca),
        ('Total NCA',               lambda s: s.total_nca),
        ('TOTAL ASSETS',            lambda s: s.total_assets),
        ('── LIABILITIES ─────────', lambda s: None),
        ('Short-term Debt',         lambda s: s.short_term_debt),
        ('Accounts Payable',        lambda s: s.accounts_payable),
        ('Taxes Payable',           lambda s: s.taxes_payable),
        ('Interest Payable',        lambda s: s.interest_payable),
        ('Lease Liab Current',      lambda s: s.lease_liab_current),
        ('Other CL',                lambda s: s.other_cl),
        ('Total Current Liab',      lambda s: s.total_cl),
        ('Long-term Debt',          lambda s: s.long_term_debt),
        ('Lease Liab NCurrent',     lambda s: s.lease_liab_noncurrent),
        ('Employee Benefits',       lambda s: s.employee_benefits),
        ('Other NCL',               lambda s: s.other_ncl),
        ('Total NCL',               lambda s: s.total_ncl),
        ('Total Liabilities',       lambda s: s.total_liabilities),
        ('── EQUITY ──────────────', lambda s: None),
        ('Share Capital',           lambda s: s.share_capital),
        ('Retained Earnings',       lambda s: s.retained_earnings),
        ('AOCI',                    lambda s: s.aoci),
        ('Total Equity',            lambda s: s.total_equity),
        ('TOTAL L+E',               lambda s: s.total_liab_equity),
        ('✓ BS diff',               lambda s: s.bs_check()[2]),
    ]
    rows = []
    for name, fn in BS_ITEMS:
        row = {'': name}
        for yr in years:
            v = fn(result.model_result.years[yr])
            row[yr] = '' if v is None else f'{v/1e6:.0f}' if abs(v) > 100 else f'{v:.2f}'
        rows.append(row)
    display(pd.DataFrame(rows).set_index(''))

## Cash Flow Statement ($M)

In [ ]:
if result.success and result.model_result:
    CF_ITEMS = [
        ('── CFO ─────────────────', lambda s: None),
        ('Net Income',              lambda s: s.cfo_net_income),
        ('D&A',                     lambda s: s.cfo_total_da),
        ('Deferred Tax',            lambda s: s.cfo_deferred_tax),
        ('Δ AR',                    lambda s: s.cfo_change_ar),
        ('Δ Inventory',             lambda s: s.cfo_change_inv),
        ('Δ AP',                    lambda s: s.cfo_change_ap),
        ('WC Delta',                lambda s: s.cfo_wc_delta),
        ('Interest paid',           lambda s: -s.cfo_interest_paid),
        ('Taxes paid',              lambda s: -s.cfo_taxes_paid),
        ('Lease payments (op)',     lambda s: s.cfo_lease_payments_operating),
        ('CFO Total',               lambda s: s.cfo_total),
        ('── CFI ─────────────────', lambda s: None),
        ('CapEx',                   lambda s: s.cfi_capex),
        ('Disposal proceeds',       lambda s: s.cfi_disposal_proceeds),
        ('CFI Total',               lambda s: s.cfi_total),
        ('── CFF ─────────────────', lambda s: None),
        ('Debt issuance',           lambda s: s.cff_debt_issuance),
        ('Debt repayment',          lambda s: s.cff_debt_repayment),
        ('Dividends',               lambda s: s.cff_dividends),
        ('Finance lease principal', lambda s: s.cff_finance_lease_principal),
        ('CFF Total',               lambda s: s.cff_total),
        ('── BRIDGE ──────────────', lambda s: None),
        ('Net Change',              lambda s: s.cf_net_change),
        ('Cash Opening',            lambda s: s.cf_cash_opening),
        ('Cash Ending',             lambda s: s.cf_cash_ending),
        ('✓ CF diff',               lambda s: s.cf_bridge_check()[2]),
    ]
    rows = []
    for name, fn in CF_ITEMS:
        row = {'': name}
        for yr in years:
            v = fn(result.model_result.years[yr])
            row[yr] = '' if v is None else f'{v/1e6:.0f}' if abs(v) > 100 else f'{v:.2f}'
        rows.append(row)
    display(pd.DataFrame(rows).set_index(''))

## Ключевые коэффициенты

In [ ]:
if result.success and result.model_result:
    KPI_ITEMS = [
        ('Revenue growth %',       lambda s, p: (s.revenue/p.revenue-1)*100 if p else None),
        ('EBITDA margin %',        lambda s, p: s.ebitda/s.revenue*100),
        ('EBIT margin %',          lambda s, p: s.ebit/s.revenue*100),
        ('NI margin %',            lambda s, p: s.net_income/s.revenue*100),
        ('D&A / Revenue %',        lambda s, p: s.total_da/s.revenue*100),
        ('CapEx / Revenue %',      lambda s, p: abs(s.cfi_capex)/s.revenue*100),
        ('Net Debt $B',            lambda s, p: (s.short_term_debt+s.long_term_debt-s.cash)/1e9),
        ('Net Debt / EBITDA',      lambda s, p: (s.short_term_debt+s.long_term_debt-s.cash)/s.ebitda if s.ebitda else None),
        ('Interest coverage (x)',  lambda s, p: s.ebit/s.interest_expense if s.interest_expense else None),
        ('FCF $M',                 lambda s, p: (s.cfo_total+s.cfi_capex)/1e6),
    ]
    rows = []
    prev_states = {yr: result.model_result.years.get(yr-1) for yr in years}
    # Базовый год как prev для первого прогнозного
    base = result.model_result  # доступ через historic.base_year_state недоступен здесь
    for name, fn in KPI_ITEMS:
        row = {'': name}
        for i, yr in enumerate(years):
            s = result.model_result.years[yr]
            p = result.model_result.years.get(yr-1)
            v = fn(s, p)
            row[yr] = '' if v is None else f'{v:.1f}'
        rows.append(row)
    display(pd.DataFrame(rows).set_index(''))

## Диагностика

In [ ]:
print('Тайминги:')
for step, t in result.timings.items():
    print(f'  {step:<20} {t:.2f}s')

if result.warnings:
    print()
    print('Предупреждения:')
    for w in result.warnings[:10]:
        print(f'  {w}')

if result.preprocess_result:
    pp = result.preprocess_result
    print()
    print(f'Препроцессор: {pp.metrics_written} метрик в {len(pp.groups_computed)} группах')
    if pp.errors:
        for e in pp.errors:
            print(f'  {e}')

# NOL tax savings
if result.success and result.model_result:
    mr = result.model_result
    STAT = 0.21
    print()
    print('NOL tax savings:')
    for yr in sorted(mr.years.keys()):
        s = mr.years[yr]
        if s.ebt > 0:
            stat = s.ebt * STAT
            actual = -s.tax_expense
            shield = stat - actual
            eff = actual / s.ebt * 100
            if shield > 1e6:
                print(f'  {yr}: shield=${shield/1e6:.0f}M  eff={eff:.1f}% (vs {STAT*100:.0f}% stat)')

# Stress results
if result.stress_results:
    print()
    print('Стресс-тест Rev D% 2025:')
    for name, sr in result.stress_results.items():
        if sr.success and 2025 in sr.comparison:
            d = sr.comparison[2025].get('revenue_delta_pct', 0)
            print(f'  {name:<25} {d:>+6.1f}%')
